# 1、PIIMiddleware中间件


In [1]:
from langchain.agents.middleware import PIIMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    #profile={"max_input_tokens":128_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

In [2]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
        PIIMiddleware("ip",strategy="block",apply_to_input=True),
    ]
)


response = agent.invoke({
    "messages" : [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

好的，我来逐步处理您的请求。不过请注意，出于安全和隐私保护，我无法直接发送邮件、查询银行余额或访问具体网站链接（包括MAC地址验证），这些操作需要您自行在对应的官方平台完成。以下是我能提供的具体指导：

---

**1. 发送邮件至 [REDACTED_EMAIL]**  
- 您可以使用自己的邮箱（如Gmail、Outlook、企业邮箱等）撰写邮件并发送。  
- 如需自动化，建议使用邮件客户端的定时发送功能，或通过脚本（如Python的`smtplib`）实现，但需确保您有权限操作该邮箱。

---

**2. 查询银行卡尾号5100的余额**  
- 请登录您银行的**官方手机APP**或**网上银行**，在账户列表中找到该卡，点击查看余额。  
- 电话银行（拨打客服热线）也可查询，但请勿在非安全环境下透露卡号。

---

**3. 访问链接 `<url_hash:dd5fc2a9>`**  
- 这是一个**哈希值**，不是完整的可访问URL。请提供完整网址（以`http://`或`https://`开头），我才能帮您分析其安全性或内容。  
- **重要提醒**：请勿点击未知来源链接，以防钓鱼或恶意软件。

---

**4. 确认MAC地址 `**-**-**-**-**-11`**  
- **MAC地址格式**：每段应为**两位十六进制数**（0-9, A-F），如 `3C-5A-B4-7E-9F-11`。  
- 您提供的部分段为`**`，缺少有效字符。请核对完整地址（共6组，每组2字符）。  
- 验证方法：在终端输入`getmac`（Windo

In [3]:
try:
    response1 = agent.invoke({
            "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到ip，抛出异常{e}")

检测到ip，抛出异常Detected 1 instance(s) of ip in text content


## 举例2：自定义检测器/函数

In [4]:
import re

# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如 "13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end() # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

In [5]:
text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
    帮我给这个号码打电话： 12345612345
    访问 https://localhost:12345
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
    帮我给这个号码打电话： ****2345
    访问 https://localhost:12345
    
================================== Ai Message ==================================

我无法帮你验证这个 API_KEY 的有效性，因为 API_KEY 通常是加密的敏感信息，需要由对应的服务提供商进行验证。我也不建议在网络上公开或分享 API_KEY。

关于打电话的部分，我无法拨打电话，但我可以帮你提供一些建议：如果你需要联系某个机构或服务，建议查看他们的官方网站或官方客服渠道获取正确号码。

至于访问 https://localhost:12345，这是你本机的地址，我无法访问。如果你需要测试本地服务，请确保本地服务器已启动，并且该端口已配置好。

如果你有其他问题，我很乐意帮忙！
